
Dataset + preprocessing (resize/normalize, light augmentation, 80/10/10 split, class weights).

Inputs/features (CNN learns hierarchical features; optional histogram equalization for a light “feature-engineering” ablation).

Model implementation (transfer learning with ResNet18, loss, optimizer, LR scheduler, early stopping, best-model saving).

Baselines (majority-class).

Evaluation (accuracy, precision, recall, F1, ROC-AUC, confusion matrix; both for CNN and baseline).

Ablations (head-only vs. “unfreeze last block” fine-tune; optional contrast equalization toggle).

In [3]:
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])


0

In [ ]:
# cnn_progress_report.py
# CNN-only pneumonia vs normal classifier (PyTorch) – progress-report ready

import os, random, math, json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
from PIL import ImageOps

# ----------------------------
# 0) Reproducibility & Config
# ----------------------------
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DATA_DIR = "chest_xray/train"   # root with NORMAL/ and PNEUMONIA/
BATCH_SIZE = 32
IMG_SIZE = 224
VAL_SPLIT = 0.10
TEST_SPLIT = 0.10
EPOCHS_PHASE_A = 6            # head-only
EPOCHS_PHASE_B = 4            # unfreeze last block
LR_HEAD = 1e-3
LR_FINETUNE = 1e-4
PATIENCE = 3                  # early stopping on val F1
USE_HIST_EQUALIZE = True     # simple feature-engineering ablation (contrast equalization)
POSITIVE_CLASS_NAME = "PNEUMONIA"  # for ROC-AUC positive class
USE_BRIGHTNESS_JITTER = True 

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT_DIR = "report_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------------------
# 1) Transforms (preprocess & augmentation)
# -----------------------------------------
class OptionalEqualize:
    """Apply histogram equalization to (grayscale) images, then convert back to 3-channel."""
    def __init__(self, enabled: bool):
        self.enabled = enabled
    def __call__(self, img):
        if not self.enabled:
            return img
        # Equalize on a single gray channel, then stack to 3 for ImageNet backbones
        gray = img.convert("L")
        eq = ImageOps.equalize(gray)
        return eq.convert("RGB")

tf_train = transforms.Compose([
    OptionalEqualize(USE_HIST_EQUALIZE),
    transforms.Grayscale(num_output_channels=3),           # dataset is X-ray (grayscale); make it 3ch for ResNet
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.2) if USE_BRIGHTNESS_JITTER else transforms.Lambda(lambda x: x),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

tf_eval = transforms.Compose([
    OptionalEqualize(USE_HIST_EQUALIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ------------------------------------------------
# 2) Load dataset + stratified 80/10/10 split
# ------------------------------------------------
# One dataset instance for label discovery (fast), then per-split datasets to attach different transforms.
discovery_ds = datasets.ImageFolder(root=DATA_DIR, transform=tf_eval)
class_to_idx = discovery_ds.class_to_idx
idx_to_class = {v:k for k,v in class_to_idx.items()}
targets = np.array([y for _, y in discovery_ds.samples])  # labels by file order

# Stratified split indices
sss_val_test = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SPLIT + TEST_SPLIT, random_state=SEED)
train_idx, valtest_idx = next(sss_val_test.split(np.zeros(len(targets)), targets))
remaining_labels = targets[valtest_idx]
rel_test = TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT)
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=rel_test, random_state=SEED)
val_idx_rel, test_idx_rel = next(sss_test.split(np.zeros(len(valtest_idx)), remaining_labels))
val_idx = valtest_idx[val_idx_rel]
test_idx = valtest_idx[test_idx_rel]

# Build per-split datasets with the proper transform
train_ds = datasets.ImageFolder(root=DATA_DIR, transform=tf_train)  # augmentation on train
val_ds   = datasets.ImageFolder(root=DATA_DIR, transform=tf_eval)
test_ds  = datasets.ImageFolder(root=DATA_DIR, transform=tf_eval)

train_set = Subset(train_ds, train_idx)
val_set   = Subset(val_ds,   val_idx)
test_set  = Subset(test_ds,  test_idx)

# Class distribution & weights (for imbalance)
train_labels = targets[train_idx]
classes, counts = np.unique(train_labels, return_counts=True)
class_counts = {int(c): int(n) for c,n in zip(classes, counts)}
class_weights = torch.tensor([len(train_labels)/class_counts[c] for c in range(len(class_to_idx))], dtype=torch.float32)
class_weights = class_weights.to(DEVICE)

# Dataloaders
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Save split stats for your Appendix
with open(os.path.join(OUT_DIR, "split_stats.json"), "w") as f:
    json.dump({
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "train_counts": class_counts,
        "n_train": len(train_idx),
        "n_val": len(val_idx),
        "n_test": len(test_idx),
        "use_hist_equalize": USE_HIST_EQUALIZE
    }, f, indent=2)

# ------------------------------------------------
# 3) Model: ResNet18 transfer learning
# ------------------------------------------------
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in model.parameters():
    p.requires_grad = False  # Phase A: head-only
num_feats = model.fc.in_features
model.fc = nn.Linear(num_feats, len(class_to_idx))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.fc.parameters(), lr=LR_HEAD)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

# ------------------------------------------------
# 4) Training / Evaluation utilities
# ------------------------------------------------
def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    y_true, y_pred, y_prob = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * yb.size(0)
        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        y_true.extend(yb.detach().cpu().numpy().tolist())
        y_pred.extend(preds.detach().cpu().numpy().tolist())
        y_prob.extend(probs.detach().cpu().numpy().tolist())
    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(y_true), np.array(y_pred), np.array(y_prob)

def compute_metrics(y_true, y_pred, y_prob, pos_index):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=pos_index, zero_division=0)
    # AUROC requires probability of the positive class
    try:
        auc = roc_auc_score((y_true==pos_index).astype(int), y_prob[:, pos_index])
    except Exception:
        auc = float('nan')
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "roc_auc": auc, "cm": cm}

def plot_curves(history):
    # Loss
    plt.figure()
    plt.plot(history["train_loss"], label="train loss")
    plt.plot(history["val_loss"], label="val loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Loss Curves"); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "loss_curves.png"), dpi=160)
    plt.close()
    # F1
    plt.figure()
    plt.plot(history["val_f1"], label="val F1")
    plt.xlabel("epoch"); plt.ylabel("F1"); plt.title("Validation F1"); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "val_f1_curve.png"), dpi=160)
    plt.close()

def plot_confusion(cm, title, filename):
    plt.figure()
    plt.imshow(cm, interpolation='nearest')
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(class_to_idx))
    plt.xticks(tick_marks, [idx_to_class[i] for i in range(len(class_to_idx))], rotation=45)
    plt.yticks(tick_marks, [idx_to_class[i] for i in range(len(class_to_idx))])
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     ha="center", va="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.ylabel('True label'); plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), dpi=160)
    plt.close()

# Positive class index (for medical recall/AUC): ensure consistent with folder naming
pos_idx = class_to_idx[POSITIVE_CLASS_NAME]

# ------------------------------------------------
# 5) Phase A: train head-only with early stopping
# ------------------------------------------------
history = {"train_loss": [], "val_loss": [], "val_f1": []}
best_val_f1 = -1
epochs_no_improve = 0

for epoch in range(EPOCHS_PHASE_A):
    tr_loss, tr_y, tr_p, tr_prob = run_epoch(train_loader, train=True)
    va_loss, va_y, va_p, va_prob = run_epoch(val_loader, train=False)
    metrics = compute_metrics(va_y, va_p, va_prob, pos_idx)
    scheduler.step(va_loss)

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(va_loss)
    history["val_f1"].append(metrics["f1"])

    print(f"[A][{epoch+1}/{EPOCHS_PHASE_A}] "
          f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} "
          f"val_acc={metrics['accuracy']:.3f} val_f1={metrics['f1']:.3f} val_rec={metrics['recall']:.3f} val_auc={metrics['roc_auc']:.3f}")

    improved = metrics["f1"] > best_val_f1 + 1e-4
    if improved:
        best_val_f1 = metrics["f1"]; epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(OUT_DIR, "best_resnet18_head.pth"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print("Early stopping (Phase A).")
            break

plot_curves(history)

# ------------------------------------------------
# 6) Phase B (optional): unfreeze last block & fine-tune
# ------------------------------------------------
# Load best head-only model first, then unfreeze last block
model.load_state_dict(torch.load(os.path.join(OUT_DIR, "best_resnet18_head.pth"), map_location=DEVICE))
for p in model.layer4.parameters():
    p.requires_grad = True
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_FINETUNE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

best_val_f1_b = -1
epochs_no_improve = 0

for epoch in range(EPOCHS_PHASE_B):
    tr_loss, tr_y, tr_p, tr_prob = run_epoch(train_loader, train=True)
    va_loss, va_y, va_p, va_prob = run_epoch(val_loader, train=False)
    metrics = compute_metrics(va_y, va_p, va_prob, pos_idx)
    scheduler.step(va_loss)

    print(f"[B][{epoch+1}/{EPOCHS_PHASE_B}] "
          f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} "
          f"val_acc={metrics['accuracy']:.3f} val_f1={metrics['f1']:.3f} val_rec={metrics['recall']:.3f} val_auc={metrics['roc_auc']:.3f}")

    improved = metrics["f1"] > best_val_f1_b + 1e-4
    if improved:
        best_val_f1_b = metrics["f1"]; epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(OUT_DIR, "best_resnet18_finetune.pth"))
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print("Early stopping (Phase B).")
            break

# Pick the best overall weights (prefer finetune if it exists)
best_path = os.path.join(OUT_DIR, "best_resnet18_finetune.pth")
if not os.path.exists(best_path):
    best_path = os.path.join(OUT_DIR, "best_resnet18_head.pth")
model.load_state_dict(torch.load(best_path, map_location=DEVICE))

# ------------------------------------------------
# 7) Majority-class baseline (simple required baseline)
# ------------------------------------------------
majority_label = int(max(class_counts, key=lambda c: class_counts[c]))
# Evaluate baseline on test set labels
test_targets = targets[test_idx]
baseline_preds = np.full_like(test_targets, fill_value=majority_label)
# dummy probs: all confidence on majority class
baseline_probs = np.zeros((len(test_targets), len(class_to_idx)))
baseline_probs[:, majority_label] = 1.0
baseline_metrics = compute_metrics(test_targets, baseline_preds, baseline_probs, pos_idx)

# ------------------------------------------------
# 8) Final Test Evaluation (CNN)
# ------------------------------------------------
te_loss, te_y, te_p, te_prob = run_epoch(test_loader, train=False)
test_metrics = compute_metrics(te_y, te_p, te_prob, pos_idx)

print("\n=== TEST RESULTS (CNN) ===")
print(json.dumps({k: (float(v) if not isinstance(v, np.ndarray) else v.tolist())
                  for k,v in test_metrics.items() if k != "cm"}, indent=2))
print("Confusion matrix (rows=true, cols=pred):\n", test_metrics["cm"])

print("\n=== BASELINE RESULTS (Majority) ===")
print(json.dumps({k: (float(v) if not isinstance(v, np.ndarray) else v.tolist())
                  for k,v in baseline_metrics.items() if k != "cm"}, indent=2))
print("Confusion matrix (rows=true, cols=pred):\n", baseline_metrics["cm"])

# Plots for the report
plot_confusion(test_metrics["cm"], "CNN Confusion Matrix (Test)", "cm_cnn_test.png")
plot_confusion(baseline_metrics["cm"], "Majority Baseline Confusion Matrix (Test)", "cm_baseline_test.png")

# Save a text report for appendix
with open(os.path.join(OUT_DIR, "classification_report_test.txt"), "w") as f:
    f.write("=== CNN (Test) ===\n")
    f.write(classification_report(te_y, te_p, target_names=[idx_to_class[i] for i in range(len(class_to_idx))]))
    f.write("\n\n=== Majority Baseline (Test) ===\n")
    f.write(classification_report(test_targets, baseline_preds, target_names=[idx_to_class[i] for i in range(len(class_to_idx))]))

print(f"\nArtifacts saved in: {OUT_DIR}\n"
      f"- best model: {best_path}\n"
      f"- split stats: split_stats.json\n"
      f"- curves: loss_curves.png, val_f1_curve.png\n"
      f"- confusion matrices: cm_cnn_test.png, cm_baseline_test.png\n"
      f"- classification report: classification_report_test.txt\n")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\luna_/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%
c:\Users\luna_\anaconda3\envs\py312\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
